In [1]:
import pandas as pd
import xgboost as xgb
import optuna
import numpy as np
from sklearn.model_selection import train_test_split, GridSearchCV, cross_val_score
from sklearn.metrics import mean_absolute_error as mae, mean_absolute_percentage_error as mape, root_mean_squared_error as rmse, r2_score

C:\Users\denis\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
pd.set_option('display.max_rows', None)

In [9]:
df = pd.read_csv('csv/stats/combine_gk_24-25.csv')
X = df[(df['Value']>0) & (df['MP']>4)]
y = X[['Value']]
X = X.drop('Value', axis=1)

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
y_binned = pd.qcut(y_train['Value'], q=5, labels=False)
X_train, X_val, y_train, y_val, _, _ = train_test_split(
    X_train, y_train, y_binned, test_size=0.2, random_state=42, stratify=y_binned
)

In [10]:
X_train[['Player', 'Nation', 'Pos', 'Squad', 'league']].to_csv('csv/final_datasets/for_xgb/train_test_val/train_gk_names.csv', index=False)
X_test[['Player', 'Nation', 'Pos', 'Squad', 'league']].to_csv('csv/final_datasets/for_xgb/train_test_val/test_gk_names.csv', index=False)
X_val[['Player', 'Nation', 'Pos', 'Squad', 'league']].to_csv('csv/final_datasets/for_xgb/train_test_val/val_gk_names.csv', index=False)

X_train.drop('Player', axis=1).to_csv('csv/final_datasets/for_xgb/train_test_val/gk_predictors_train.csv', index=False)
X_test.drop('Player', axis=1).to_csv('csv/final_datasets/for_xgb/train_test_val/gk_predictors_test.csv', index=False)
X_val.drop('Player', axis=1).to_csv('csv/final_datasets/for_xgb/train_test_val/gk_predictors_val.csv', index=False)

y_train.to_csv('csv/final_datasets/for_xgb/train_test_val/gk_target_train.csv', index=False)
y_test.to_csv('csv/final_datasets/for_xgb/train_test_val/gk_target_test.csv', index=False)
y_val.to_csv('csv/final_datasets/for_xgb/train_test_val/gk_target_val.csv', index=False)

In [11]:
X_train = pd.read_csv('csv/final_datasets/for_xgb/train_test_val/gk_predictors_train.csv')
X_test = pd.read_csv('csv/final_datasets/for_xgb/train_test_val/gk_predictors_test.csv')
X_val = pd.read_csv('csv/final_datasets/for_xgb/train_test_val/gk_predictors_val.csv')
for col in ('Nation', 'Pos', 'Squad', 'league'):
    X_train[col] = X_train[col].astype('category')
    X_test[col] = X_test[col].astype('category')
    X_val[col] = X_val[col].astype('category')

players_train = pd.read_csv('csv/final_datasets/for_xgb/train_test_val/train_gk_names.csv')
players_test = pd.read_csv('csv/final_datasets/for_xgb/train_test_val/test_gk_names.csv')
players_val = pd.read_csv('csv/final_datasets/for_xgb/train_test_val/val_gk_names.csv')

y_train = pd.read_csv('csv/final_datasets/for_xgb/train_test_val/gk_target_train.csv')
y_test = pd.read_csv('csv/final_datasets/for_xgb/train_test_val/gk_target_test.csv')
y_val = pd.read_csv('csv/final_datasets/for_xgb/train_test_val/gk_target_val.csv')

In [12]:
def objective_gamma(trial):
    params = {
        'max_depth' : trial.suggest_int('max_depth', 3, 10),
        'learning_rate' : trial.suggest_float('learning_rate', 0.01, 10, log=True),
        'subsample' : trial.suggest_float('subsample', 0.5, 1),
        'colsample_bytree' : trial.suggest_float('colsample_bytree', 0.5, 1),
        'reg_lambda' : trial.suggest_float('reg_lambda', 0.01, 10, log=True),
        'reg_alpha' : trial.suggest_float('reg_alpha', 0.01, 10, log=True)
    }

    params['n_estimators'] = 2000
    params['objective'] = 'reg:gamma'
    params['random_state'] = 42
    #params['device'] = 'cuda'
    params['early_stopping_rounds'] = 15
    params['eval_metric'] = 'mape'
    params['enable_categorical'] = True

    model = xgb.XGBRegressor(**params)
    model.fit(
        X_train, y_train,
        eval_set = [(X_val, y_val)],
        verbose = False
    )

    y_val_pred = model.predict(X_val)
    y_train_pred = model.predict(X_train)

    r2_val = r2_score(y_val, y_val_pred)
    r2_train = r2_score(y_train, y_train_pred)

    mape_val = mape(y_val, y_val_pred)
    
    trial.set_user_attr('r2_train', r2_train)
    trial.set_user_attr('r2_val', r2_val)
    trial.set_user_attr('r2_delta', abs(r2_train - r2_val))
    trial.set_user_attr('mape_val', mape_val)
    
    return r2_val

In [13]:
study = optuna.create_study(direction='maximize', study_name='xgb_tuning_gamma')

[I 2026-04-05 15:31:52,480] A new study created in memory with name: xgb_tuning_gamma


In [14]:
study.optimize(objective_gamma, n_trials=100, show_progress_bar=True, n_jobs=-1)

Best trial: 6. Best value: 0.126267:   5%|██▎                                          | 5/100 [00:00<00:09, 10.54it/s]

[I 2026-04-05 15:32:14,450] Trial 2 finished with value: -76.37210083007812 and parameters: {'max_depth': 5, 'learning_rate': 4.830652710983124, 'subsample': 0.7649186754706936, 'colsample_bytree': 0.7068289090056487, 'reg_lambda': 0.7725826039793499, 'reg_alpha': 2.1411834574438036}. Best is trial 2 with value: -76.37210083007812.
[I 2026-04-05 15:32:14,467] Trial 1 finished with value: -7.026104927062988 and parameters: {'max_depth': 8, 'learning_rate': 2.98539903688901, 'subsample': 0.8934818921922802, 'colsample_bytree': 0.8981826523540128, 'reg_lambda': 1.270637275304875, 'reg_alpha': 0.015498440370893223}. Best is trial 1 with value: -7.026104927062988.
[I 2026-04-05 15:32:14,497] Trial 6 finished with value: 0.12626659870147705 and parameters: {'max_depth': 4, 'learning_rate': 0.5007968887071863, 'subsample': 0.7036487314844136, 'colsample_bytree': 0.5798117553881201, 'reg_lambda': 0.021566445970805266, 'reg_alpha': 1.7931037257787208}. Best is trial 6 with value: 0.126266598701

Best trial: 6. Best value: 0.126267:   8%|███▌                                         | 8/100 [00:00<00:09, 10.13it/s]

[I 2026-04-05 15:32:14,760] Trial 12 finished with value: -1.522695779800415 and parameters: {'max_depth': 4, 'learning_rate': 2.203267375984796, 'subsample': 0.6837980211686431, 'colsample_bytree': 0.7379002804441612, 'reg_lambda': 1.7745733107466681, 'reg_alpha': 0.467562210825681}. Best is trial 6 with value: 0.12626659870147705.
[I 2026-04-05 15:32:14,841] Trial 0 finished with value: 0.02878338098526001 and parameters: {'max_depth': 9, 'learning_rate': 0.060921370568431714, 'subsample': 0.8994598626421896, 'colsample_bytree': 0.5462119135524026, 'reg_lambda': 1.2016844384633467, 'reg_alpha': 0.048781161371075}. Best is trial 6 with value: 0.12626659870147705.
[I 2026-04-05 15:32:14,874] Trial 3 finished with value: 0.10561317205429077 and parameters: {'max_depth': 9, 'learning_rate': 0.05652466179711447, 'subsample': 0.7547879894302172, 'colsample_bytree': 0.7716113097555641, 'reg_lambda': 1.6992337255015595, 'reg_alpha': 0.42333655235980544}. Best is trial 6 with value: 0.1262665

Best trial: 7. Best value: 0.3517:  13%|█████▉                                        | 13/100 [00:01<00:06, 13.71it/s]

...

[I 2026-04-05 15:32:25,324] Trial 92 finished with value: 0.5278145670890808 and parameters: {'max_depth': 3, 'learning_rate': 0.03145701157327496, 'subsample': 0.9765757501501688, 'colsample_bytree': 0.977787378192523, 'reg_lambda': 7.427211941425288, 'reg_alpha': 2.0724146451180836}. Best is trial 82 with value: 0.5479058027267456.
[I 2026-04-05 15:32:25,386] Trial 96 finished with value: 0.47882670164108276 and parameters: {'max_depth': 3, 'learning_rate': 0.0336393708076762, 'subsample': 0.9400138084768216, 'colsample_bytree': 0.9773697177922274, 'reg_lambda': 6.94750891071871, 'reg_alpha': 6.708809849108654}. Best is trial 82 with value: 0.5479058027267456.


Best trial: 82. Best value: 0.547906:  95%|████████████████████████████████████████▊  | 95/100 [00:11<00:00,  6.50it/s]

[I 2026-04-05 15:32:25,575] Trial 98 finished with value: 0.49226653575897217 and parameters: {'max_depth': 3, 'learning_rate': 0.03505491816600592, 'subsample': 0.9748239737439314, 'colsample_bytree': 0.9773317076853837, 'reg_lambda': 6.7992406566373695, 'reg_alpha': 2.00277593483615}. Best is trial 82 with value: 0.5479058027267456.
[I 2026-04-05 15:32:25,630] Trial 97 finished with value: 0.4985840320587158 and parameters: {'max_depth': 3, 'learning_rate': 0.035377779743061735, 'subsample': 0.8311356427358032, 'colsample_bytree': 0.9757847174881805, 'reg_lambda': 5.928509682293866, 'reg_alpha': 2.0799324444313934}. Best is trial 82 with value: 0.5479058027267456.


Best trial: 82. Best value: 0.547906: 100%|██████████████████████████████████████████| 100/100 [00:11<00:00,  8.37it/s]

[I 2026-04-05 15:32:25,890] Trial 95 finished with value: 0.515710175037384 and parameters: {'max_depth': 3, 'learning_rate': 0.012478075408854827, 'subsample': 0.9715027472455198, 'colsample_bytree': 0.9784476332515112, 'reg_lambda': 7.061095965593016, 'reg_alpha': 1.9450237700996809}. Best is trial 82 with value: 0.5479058027267456.
[I 2026-04-05 15:32:25,945] Trial 93 finished with value: 0.4324914813041687 and parameters: {'max_depth': 3, 'learning_rate': 0.0102638711625892, 'subsample': 0.9734386736107682, 'colsample_bytree': 0.9773728815640771, 'reg_lambda': 7.089336936576464, 'reg_alpha': 7.438654914108618}. Best is trial 82 with value: 0.5479058027267456.
[I 2026-04-05 15:32:26,014] Trial 99 finished with value: 0.4915968179702759 and parameters: {'max_depth': 3, 'learning_rate': 0.011987598361689304, 'subsample': 0.9729465426709692, 'colsample_bytree': 0.9697769979486693, 'reg_lambda': 7.430776109446629, 'reg_alpha': 2.0722229995116095}. Best is trial 82 with value: 0.54790580

In [20]:
valid_trials = []
for trial in study.trials:
    if trial.state != optuna.trial.TrialState.COMPLETE:
        continue
    r2_val = trial.user_attrs['r2_val']
    if r2_val > 0.5:
        valid_trials.append(trial)

In [21]:
if len(valid_trials) > 0:
    valid_trials.sort(key=lambda t:t.user_attrs['r2_delta'])

In [22]:
best_trials = valid_trials[:15]

In [23]:
best_trials_df = pd.DataFrame({
    'r2_val':[trial.user_attrs['r2_val'] for trial in best_trials],
    'r2_train':[trial.user_attrs['r2_train'] for trial in best_trials],
    'r2_delta':[trial.user_attrs['r2_delta'] for trial in best_trials],
    'mape_val':[trial.user_attrs['mape_val'] for trial in best_trials]
})

In [24]:
best_trials_df

,r2_val,r2_train,r2_delta,mape_val
0,0.519855,0.724727,0.204872,1.069056
1,0.512601,0.720019,0.207418,0.941005
2,0.528987,0.742944,0.213957,1.061981
3,0.512143,0.727861,0.215718,1.062255
4,0.521937,0.744028,0.222090,1.093129
5,0.500169,0.728887,0.228717,1.074196
6,0.546095,0.776080,0.229984,1.092074
7,0.530674,0.762458,0.231785,1.000961
8,0.547906,0.782956,0.235050,0.979542
9,0.521587,0.764314,0.242727,0.995608


In [137]:
params = best_trials[9].params
params['n_estimators'] = 2000
params['objective'] = 'reg:gamma'
params['random_state'] = 42
#params['device'] = 'cuda'
params['early_stopping_rounds'] = 15
params['eval_metric'] = 'mape'
params['enable_categorical'] = True

In [138]:
xgb_regressor = xgb.XGBRegressor(**params)
xgb_regressor.fit(
    X_train, y_train,
    eval_set = [(X_val, y_val)],
    verbose = True
)

[0]	validation_0-mape:3.05749
[1]	validation_0-mape:2.97986
[2]	validation_0-mape:2.90074
[3]	validation_0-mape:2.86854
[4]	validation_0-mape:2.80705
[5]	validation_0-mape:2.76910
[6]	validation_0-mape:2.72971
[7]	validation_0-mape:2.70741
[8]	validation_0-mape:2.67377
[9]	validation_0-mape:2.61581
[10]	validation_0-mape:2.57014
[11]	validation_0-mape:2.54305
[12]	validation_0-mape:2.50467
[13]	validation_0-mape:2.46773
[14]	validation_0-mape:2.39964
[15]	validation_0-mape:2.36442
[16]	validation_0-mape:2.31763
[17]	validation_0-mape:2.29820
[18]	validation_0-mape:2.27423
[19]	validation_0-mape:2.25347
[20]	validation_0-mape:2.22653
[21]	validation_0-mape:2.20927
[22]	validation_0-mape:2.18148
[23]	validation_0-mape:2.16280
[24]	validation_0-mape:2.12223
[25]	validation_0-mape:2.10558
[26]	validation_0-mape:2.09097
[27]	validation_0-mape:2.07232
[28]	validation_0-mape:2.04386
[29]	validation_0-mape:2.02445
[30]	validation_0-mape:2.01109
[31]	validation_0-mape:1.98758
[32]	validation_0-

,"objective objective: str | xgboost.sklearn._SklObjWProto | typing.Callable[[typing.Any, typing.Any], typing.Tuple[numpy.ndarray, numpy.ndarray]] | NoneSpecify the learning task and the corresponding learning objective or a customobjective function to be used.For custom objective, see :doc:`/tutorials/custom_metric_obj` and:ref:`custom-obj-metric` for more information, along with the end note forfunction signatures.",'reg:gamma'
,"base_score base_score: float | typing.List[float] | NoneThe initial prediction score of all instances, global bias.",None
,booster,None
,"callbacks callbacks: typing.List[xgboost.callback.TrainingCallback] | NoneList of callback functions that are applied at end of each iteration.It is possible to use predefined callbacks by using:ref:`Callback API `... note:: States in callback are not preserved during training, which means callback objects can not be reused for multiple training sessions without reinitialization or deepcopy... code-block:: python for params in parameters_grid: # be sure to (re)initialize the callbacks before each run callbacks = [xgb.callback.LearningRateScheduler(custom_rates)] reg = xgboost.XGBRegressor(**params, callbacks=callbacks) reg.fit(X, y)",None
,colsample_bylevel colsample_bylevel: float | NoneSubsample ratio of columns for each level.,None
,colsample_bynode colsample_bynode: float | NoneSubsample ratio of columns for each split.,None
,colsample_bytree colsample_bytree: float | NoneSubsample ratio of columns when constructing each tree.,0.9991000631428529
,"device device: str | None.. versionadded:: 2.0.0Device ordinal, available options are `cpu`, `cuda`, and `gpu`.",None
,"early_stopping_rounds early_stopping_rounds: int | None.. versionadded:: 1.6.0- Activates early stopping. Validation metric needs to improve at least once in every **early_stopping_rounds** round(s) to continue training. Requires at least one item in **eval_set** in :py:meth:`fit`.- If early stopping occurs, the model will have two additional attributes: :py:attr:`best_score` and :py:attr:`best_iteration`. These are used by the :py:meth:`predict` and :py:meth:`apply` methods to determine the optimal number of trees during inference. If users want to access the full model (including trees built after early stopping), they can specify the `iteration_range` in these inference methods. In addition, other utilities like model plotting can also use the entire model.- If you prefer to discard the trees after `best_iteration`, consider using the callback function :py:class:`xgboost.callback.EarlyStopping`.- If there's more than one item in **eval_set**, the last entry will be used for early stopping. If there's more than one metric in **eval_metric**, the last metric will be used for early stopping.",15
,enable_categorical enable_categorical: boolSee the same parameter of :py:class:`DMatrix` for details.,True
,"eval_metric eval_metric: str | typing.List[str | typing.Callable] | typing.Callable | None.. versionadded:: 1.6.0Metric used for monitoring the training result and early stopping. It can be astring or list of strings as names of predefined metric in XGBoost (See:doc:`/parameter`), one of the metrics in :py:mod:`sklearn.metrics`, or anyother user defined metric that looks like `sklearn.metrics`.If custom objective is also provided, then custom metric should implement thecorresponding reverse link function.Unlike the `scoring` parameter commonly used in scikit-learn, when a callableobject is provided, it's assumed to be a cost function and by default XGBoostwill minimize the result during early stopping.For advanced usage on Early stopping like directly choosing to maximize insteadof minimize, see :py:obj:`xgboost.callback.EarlyStopping`.See :doc:`/tutorials/custom_metric_obj` and :ref:`custom-obj-metric` for moreinformation... code-block:: python from sklearn.datasets import load_diabetes from sklearn.metrics import mean_absolute_error X, y = load_diabetes(return_X_y=True) reg = xgb.XGBRegressor( tree

In [139]:
y_train_pred = xgb_regressor.predict(X_train)
y_test_pred = xgb_regressor.predict(X_test)
y_val_pred = xgb_regressor.predict(X_val)

In [140]:
response_train = pd.concat([players_train, pd.Series(y_train_pred, name='Value_pred'), y_train], axis=1)
response_test = pd.concat([players_test, pd.Series(y_test_pred, name='Value_pred'), y_test], axis=1)
response_val = pd.concat([players_val, pd.Series(y_val_pred, name='Value_pred'), y_val], axis=1)

In [141]:
response_train_val = pd.concat([response_train, response_val])
response = pd.concat([response_train_val, response_test])

In [142]:
metrics = pd.DataFrame({
    'MAE':[mae(y_test_pred, y_test), mae(y_train_pred, y_train), mae(y_val_pred, y_val)],
    'RMSE':[rmse(y_test_pred, y_test), rmse(y_train_pred, y_train), rmse(y_val_pred, y_val)],
    'MAPE':[mape(y_test_pred, y_test), mape(y_train_pred, y_train), mape(y_val_pred, y_val)],
    'R2': [r2_score(y_test_pred, y_test), r2_score(y_train_pred, y_train), r2_score(y_val_pred, y_val)]
}, index = ['test', 'train', 'val'])

In [143]:
metrics

,MAE,RMSE,MAPE,R2
test,2.590815,4.189306,0.607201,0.203410
train,2.150496,4.232806,0.388970,0.412158
val,3.213408,5.287226,0.530461,-1.404599


In [106]:
response.sort_values('Value_pred', ascending=False)

,Player,Nation,Pos,Squad,league,Value_pred,Value
84,Gianluigi Donnarumma,ITA,GK,Paris Saint-Germain,FR1,24.282331,40.00
97,Robert Sanchez,ESP,GK,Chelsea,GB1,23.928696,20.00
95,David Raya,ESP,GK,Arsenal,GB1,23.896221,40.00
101,Mile Svilar,SRB,GK,Roma,IT1,22.049604,25.00
158,Lucas Chevalier,FRA,GK,Lille,FR1,21.958185,30.00
30,Marco Carnesecchi,ITA,GK,Atalanta,IT1,21.269358,25.00
25,Alex Meret,ITA,GK,Napoli,IT1,19.649870,18.00
15,Michele Di Gregorio,ITA,GK,Juventus,IT1,19.183144,18.00
150,Anatolii Trubin,UKR,GK,Benfica,PO1,18.487362,28.00
50,Diogo Costa,POR,GK,Porto,PO1,18.429121,40.00


In [92]:
params = best_trials[8].params
params['n_estimators'] = 2000
params['objective'] = 'reg:gamma'
params['random_state'] = 42
#params['device'] = 'cuda'
params['early_stopping_rounds'] = 15
params['eval_metric'] = 'mape'
params['enable_categorical'] = True

In [93]:
xgb_regressor = xgb.XGBRegressor(**params)
xgb_regressor.fit(
    X_train, y_train,
    eval_set = [(X_val, y_val)],
    verbose = True
)

[0]	validation_0-mape:3.05882
[1]	validation_0-mape:2.98085
[2]	validation_0-mape:2.89314
[3]	validation_0-mape:2.86183
[4]	validation_0-mape:2.79311
[5]	validation_0-mape:2.75633
[6]	validation_0-mape:2.71390
[7]	validation_0-mape:2.67799
[8]	validation_0-mape:2.64064
[9]	validation_0-mape:2.59289
[10]	validation_0-mape:2.54246
[11]	validation_0-mape:2.52138
[12]	validation_0-mape:2.47466
[13]	validation_0-mape:2.43324
[14]	validation_0-mape:2.35934
[15]	validation_0-mape:2.32172
[16]	validation_0-mape:2.29133
[17]	validation_0-mape:2.25983
[18]	validation_0-mape:2.23614
[19]	validation_0-mape:2.21240
[20]	validation_0-mape:2.18930
[21]	validation_0-mape:2.16685
[22]	validation_0-mape:2.13881
[23]	validation_0-mape:2.12363
[24]	validation_0-mape:2.08754
[25]	validation_0-mape:2.06530
[26]	validation_0-mape:2.03631
[27]	validation_0-mape:2.00653
[28]	validation_0-mape:1.97784
[29]	validation_0-mape:1.95832
[30]	validation_0-mape:1.94551
[31]	validation_0-mape:1.91961
[32]	validation_0-

,"objective objective: str | xgboost.sklearn._SklObjWProto | typing.Callable[[typing.Any, typing.Any], typing.Tuple[numpy.ndarray, numpy.ndarray]] | NoneSpecify the learning task and the corresponding learning objective or a customobjective function to be used.For custom objective, see :doc:`/tutorials/custom_metric_obj` and:ref:`custom-obj-metric` for more information, along with the end note forfunction signatures.",'reg:gamma'
,"base_score base_score: float | typing.List[float] | NoneThe initial prediction score of all instances, global bias.",None
,booster,None
,"callbacks callbacks: typing.List[xgboost.callback.TrainingCallback] | NoneList of callback functions that are applied at end of each iteration.It is possible to use predefined callbacks by using:ref:`Callback API `... note:: States in callback are not preserved during training, which means callback objects can not be reused for multiple training sessions without reinitialization or deepcopy... code-block:: python for params in parameters_grid: # be sure to (re)initialize the callbacks before each run callbacks = [xgb.callback.LearningRateScheduler(custom_rates)] reg = xgboost.XGBRegressor(**params, callbacks=callbacks) reg.fit(X, y)",None
,colsample_bylevel colsample_bylevel: float | NoneSubsample ratio of columns for each level.,None
,colsample_bynode colsample_bynode: float | NoneSubsample ratio of columns for each split.,None
,colsample_bytree colsample_bytree: float | NoneSubsample ratio of columns when constructing each tree.,0.9996129777967127
,"device device: str | None.. versionadded:: 2.0.0Device ordinal, available options are `cpu`, `cuda`, and `gpu`.",None
,"early_stopping_rounds early_stopping_rounds: int | None.. versionadded:: 1.6.0- Activates early stopping. Validation metric needs to improve at least once in every **early_stopping_rounds** round(s) to continue training. Requires at least one item in **eval_set** in :py:meth:`fit`.- If early stopping occurs, the model will have two additional attributes: :py:attr:`best_score` and :py:attr:`best_iteration`. These are used by the :py:meth:`predict` and :py:meth:`apply` methods to determine the optimal number of trees during inference. If users want to access the full model (including trees built after early stopping), they can specify the `iteration_range` in these inference methods. In addition, other utilities like model plotting can also use the entire model.- If you prefer to discard the trees after `best_iteration`, consider using the callback function :py:class:`xgboost.callback.EarlyStopping`.- If there's more than one item in **eval_set**, the last entry will be used for early stopping. If there's more than one metric in **eval_metric**, the last metric will be used for early stopping.",15
,enable_categorical enable_categorical: boolSee the same parameter of :py:class:`DMatrix` for details.,True
,"eval_metric eval_metric: str | typing.List[str | typing.Callable] | typing.Callable | None.. versionadded:: 1.6.0Metric used for monitoring the training result and early stopping. It can be astring or list of strings as names of predefined metric in XGBoost (See:doc:`/parameter`), one of the metrics in :py:mod:`sklearn.metrics`, or anyother user defined metric that looks like `sklearn.metrics`.If custom objective is also provided, then custom metric should implement thecorresponding reverse link function.Unlike the `scoring` parameter commonly used in scikit-learn, when a callableobject is provided, it's assumed to be a cost function and by default XGBoostwill minimize the result during early stopping.For advanced usage on Early stopping like directly choosing to maximize insteadof minimize, see :py:obj:`xgboost.callback.EarlyStopping`.See :doc:`/tutorials/custom_metric_obj` and :ref:`custom-obj-metric` for moreinformation... code-block:: python from sklearn.datasets import load_diabetes from sklearn.metrics import mean_absolute_error X, y = load_diabetes(return_X_y=True) reg = xgb.XGBRegressor( tree

In [94]:
y_train_pred = xgb_regressor.predict(X_train)
y_test_pred = xgb_regressor.predict(X_test)
y_val_pred = xgb_regressor.predict(X_val)

In [95]:
response_train = pd.concat([players_train, pd.Series(y_train_pred, name='Value_pred'), y_train], axis=1)
response_test = pd.concat([players_test, pd.Series(y_test_pred, name='Value_pred'), y_test], axis=1)
response_val = pd.concat([players_val, pd.Series(y_val_pred, name='Value_pred'), y_val], axis=1)

In [96]:
response_train_val = pd.concat([response_train, response_val])
response = pd.concat([response_train_val, response_test])

In [97]:
metrics = pd.DataFrame({
    'MAE':[mae(y_test_pred, y_test), mae(y_train_pred, y_train), mae(y_val_pred, y_val)],
    'RMSE':[rmse(y_test_pred, y_test), rmse(y_train_pred, y_train), rmse(y_val_pred, y_val)],
    'MAPE':[mape(y_test_pred, y_test), mape(y_train_pred, y_train), mape(y_val_pred, y_val)],
    'R2': [r2_score(y_test_pred, y_test), r2_score(y_train_pred, y_train), r2_score(y_val_pred, y_val)]
}, index = ['test', 'train', 'val'])

In [98]:
metrics

,MAE,RMSE,MAPE,R2
test,2.507381,4.052343,0.601795,0.287945
train,2.066061,4.061953,0.369476,0.491414
val,3.137590,5.139736,0.509731,-1.059596


In [83]:
params = best_trials[7].params
params['n_estimators'] = 2000
params['objective'] = 'reg:gamma'
params['random_state'] = 42
#params['device'] = 'cuda'
params['early_stopping_rounds'] = 15
params['eval_metric'] = 'mape'
params['enable_categorical'] = True

In [84]:
xgb_regressor = xgb.XGBRegressor(**params)
xgb_regressor.fit(
    X_train, y_train,
    eval_set = [(X_val, y_val)],
    verbose = True
)

[0]	validation_0-mape:3.04204
[1]	validation_0-mape:2.95223
[2]	validation_0-mape:2.86233
[3]	validation_0-mape:2.82966
[4]	validation_0-mape:2.76129
[5]	validation_0-mape:2.72205
[6]	validation_0-mape:2.67613
[7]	validation_0-mape:2.65213
[8]	validation_0-mape:2.61583
[9]	validation_0-mape:2.55351
[10]	validation_0-mape:2.50193
[11]	validation_0-mape:2.47575
[12]	validation_0-mape:2.43461
[13]	validation_0-mape:2.39489
[14]	validation_0-mape:2.31783
[15]	validation_0-mape:2.28124
[16]	validation_0-mape:2.24896
[17]	validation_0-mape:2.21941
[18]	validation_0-mape:2.19720
[19]	validation_0-mape:2.17320
[20]	validation_0-mape:2.14579
[21]	validation_0-mape:2.11319
[22]	validation_0-mape:2.08591
[23]	validation_0-mape:2.07089
[24]	validation_0-mape:2.03127
[25]	validation_0-mape:2.01355
[26]	validation_0-mape:1.98550
[27]	validation_0-mape:1.96840
[28]	validation_0-mape:1.93720
[29]	validation_0-mape:1.91359
[30]	validation_0-mape:1.88472
[31]	validation_0-mape:1.86070
[32]	validation_0-

,"objective objective: str | xgboost.sklearn._SklObjWProto | typing.Callable[[typing.Any, typing.Any], typing.Tuple[numpy.ndarray, numpy.ndarray]] | NoneSpecify the learning task and the corresponding learning objective or a customobjective function to be used.For custom objective, see :doc:`/tutorials/custom_metric_obj` and:ref:`custom-obj-metric` for more information, along with the end note forfunction signatures.",'reg:gamma'
,"base_score base_score: float | typing.List[float] | NoneThe initial prediction score of all instances, global bias.",None
,booster,None
,"callbacks callbacks: typing.List[xgboost.callback.TrainingCallback] | NoneList of callback functions that are applied at end of each iteration.It is possible to use predefined callbacks by using:ref:`Callback API `... note:: States in callback are not preserved during training, which means callback objects can not be reused for multiple training sessions without reinitialization or deepcopy... code-block:: python for params in parameters_grid: # be sure to (re)initialize the callbacks before each run callbacks = [xgb.callback.LearningRateScheduler(custom_rates)] reg = xgboost.XGBRegressor(**params, callbacks=callbacks) reg.fit(X, y)",None
,colsample_bylevel colsample_bylevel: float | NoneSubsample ratio of columns for each level.,None
,colsample_bynode colsample_bynode: float | NoneSubsample ratio of columns for each split.,None
,colsample_bytree colsample_bytree: float | NoneSubsample ratio of columns when constructing each tree.,0.9950705474817999
,"device device: str | None.. versionadded:: 2.0.0Device ordinal, available options are `cpu`, `cuda`, and `gpu`.",None
,"early_stopping_rounds early_stopping_rounds: int | None.. versionadded:: 1.6.0- Activates early stopping. Validation metric needs to improve at least once in every **early_stopping_rounds** round(s) to continue training. Requires at least one item in **eval_set** in :py:meth:`fit`.- If early stopping occurs, the model will have two additional attributes: :py:attr:`best_score` and :py:attr:`best_iteration`. These are used by the :py:meth:`predict` and :py:meth:`apply` methods to determine the optimal number of trees during inference. If users want to access the full model (including trees built after early stopping), they can specify the `iteration_range` in these inference methods. In addition, other utilities like model plotting can also use the entire model.- If you prefer to discard the trees after `best_iteration`, consider using the callback function :py:class:`xgboost.callback.EarlyStopping`.- If there's more than one item in **eval_set**, the last entry will be used for early stopping. If there's more than one metric in **eval_metric**, the last metric will be used for early stopping.",15
,enable_categorical enable_categorical: boolSee the same parameter of :py:class:`DMatrix` for details.,True
,"eval_metric eval_metric: str | typing.List[str | typing.Callable] | typing.Callable | None.. versionadded:: 1.6.0Metric used for monitoring the training result and early stopping. It can be astring or list of strings as names of predefined metric in XGBoost (See:doc:`/parameter`), one of the metrics in :py:mod:`sklearn.metrics`, or anyother user defined metric that looks like `sklearn.metrics`.If custom objective is also provided, then custom metric should implement thecorresponding reverse link function.Unlike the `scoring` parameter commonly used in scikit-learn, when a callableobject is provided, it's assumed to be a cost function and by default XGBoostwill minimize the result during early stopping.For advanced usage on Early stopping like directly choosing to maximize insteadof minimize, see :py:obj:`xgboost.callback.EarlyStopping`.See :doc:`/tutorials/custom_metric_obj` and :ref:`custom-obj-metric` for moreinformation... code-block:: python from sklearn.datasets import load_diabetes from sklearn.metrics import mean_absolute_error X, y = load_diabetes(return_X_y=True) reg = xgb.XGBRegressor( tree

In [85]:
y_train_pred = xgb_regressor.predict(X_train)
y_test_pred = xgb_regressor.predict(X_test)
y_val_pred = xgb_regressor.predict(X_val)

In [86]:
response_train = pd.concat([players_train, pd.Series(y_train_pred, name='Value_pred'), y_train], axis=1)
response_test = pd.concat([players_test, pd.Series(y_test_pred, name='Value_pred'), y_test], axis=1)
response_val = pd.concat([players_val, pd.Series(y_val_pred, name='Value_pred'), y_val], axis=1)

In [87]:
response_train_val = pd.concat([response_train, response_val])
response = pd.concat([response_train_val, response_test])

In [88]:
metrics = pd.DataFrame({
    'MAE':[mae(y_test_pred, y_test), mae(y_train_pred, y_train), mae(y_val_pred, y_val)],
    'RMSE':[rmse(y_test_pred, y_test), rmse(y_train_pred, y_train), rmse(y_val_pred, y_val)],
    'MAPE':[mape(y_test_pred, y_test), mape(y_train_pred, y_train), mape(y_val_pred, y_val)],
    'R2': [r2_score(y_test_pred, y_test), r2_score(y_train_pred, y_train), r2_score(y_val_pred, y_val)]
}, index = ['test', 'train', 'val'])

In [89]:
metrics

,MAE,RMSE,MAPE,R2
test,2.524376,4.110988,0.600594,0.208191
train,2.135694,4.249435,0.382751,0.393346
val,3.202610,5.236773,0.517033,-1.280099


In [91]:
response.sort_values('Value_pred', ascending=False)

,Player,Nation,Pos,Squad,league,Value_pred,Value
84,Gianluigi Donnarumma,ITA,GK,Paris Saint-Germain,FR1,24.914911,40.00
95,David Raya,ESP,GK,Arsenal,GB1,23.349607,40.00
97,Robert Sanchez,ESP,GK,Chelsea,GB1,22.712318,20.00
101,Mile Svilar,SRB,GK,Roma,IT1,21.492401,25.00
30,Marco Carnesecchi,ITA,GK,Atalanta,IT1,21.032005,25.00
158,Lucas Chevalier,FRA,GK,Lille,FR1,20.985502,30.00
150,Anatolii Trubin,UKR,GK,Benfica,PO1,18.694334,28.00
15,Michele Di Gregorio,ITA,GK,Juventus,IT1,18.624432,18.00
25,Alex Meret,ITA,GK,Napoli,IT1,18.524477,18.00
71,Guglielmo Vicario,ITA,GK,Tottenham Hotspur,GB1,18.371302,32.00
